# Vaca Muerta — Notebook maestro integral · Producción de petróleo y Prophet 2026
**Provincia del Neuquén · Formación Vaca Muerta · recurso SHALE · petróleo**  
Autor: Lic. en Ciencias Geológicas Gustavo Herrero · Cierre editorial 08/10/2026.

Este Notebook contiene el flujo reproducible **desde el Excel mensual consolidado** hasta el pronóstico 2026. El filtrado y la normalización originales de los CSV nacionales anuales fueron efectuados previamente y sus resultados están documentados en las hojas `Control_calidad` y `Criterios`; **no se incluyen los CSV brutos, ni se simula repetir su depuración desde datos por pozo**.

> **Reproducibilidad:** este Notebook se entrega con celdas sin ejecutar. El valor 35.596.087,84 m³ procede de la ejecución final informada el 06/10/2026 y se utiliza únicamente como *referencia de conciliación*. El modelo debe ejecutarse de nuevo para verificar si el reentrenamiento 2020–2025 lo reproduce. El diseño usa test 2024/2025 sin fuga de información temporal, y una configuración final fija CPS=0,03/SPS=0,1.

## 0. Entorno y archivos
Colocar este `.ipynb` y el Excel adjunto en la misma carpeta. Ejecutar las celdas en orden. Instalar `prophet`, `pandas`, `numpy`, `matplotlib`, `scikit-learn`, `statsmodels`, `openpyxl` en el entorno de Jupyter cuando corresponda.

In [ ]:
# Instalación solo si alguna dependencia falta (ejecutar en una celda aparte):
# %pip install prophet pandas numpy matplotlib scikit-learn statsmodels openpyxl
from pathlib import Path
import sys, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.stats.diagnostic import acorr_ljungbox
from IPython.display import display

ARCHIVO = Path('VM_Serie_Fuente_2012_2025.xlsx')
SALIDA = Path('salidas_VM_2026')
SALIDA.mkdir(exist_ok=True)
print('Python:', sys.version.split()[0], ' | Pandas:',pd.__version__)
print('Fuente:', ARCHIVO.resolve())
assert ARCHIVO.exists(), 'No se encuentra el Excel de la serie consolidada junto al Notebook.' 

## 1. Lectura de la serie y trazabilidad
La hoja `Serie_mensual` contiene la serie lista para modelar; la hoja `Serie_anual` y `Control_calidad` se usan para controles. Los registros originales individuales por pozo no se reconstruyen en este Notebook.

In [ ]:
xls = pd.ExcelFile(ARCHIVO)
print('Hojas:', xls.sheet_names)
assert {'Serie_mensual','Serie_anual','Control_calidad','Criterios'}.issubset(xls.sheet_names)
base = pd.read_excel(ARCHIVO, sheet_name='Serie_mensual')
print('Columnas:',base.columns.tolist())
df = base[['Fecha','Producción petróleo (m³)']].copy()
df.columns = ['ds','y']
df['ds'] = pd.to_datetime(df['ds'].astype(str), format='%Y-%m', errors='raise')
df['y'] = pd.to_numeric(df['y'], errors='raise')
df = df.sort_values('ds').reset_index(drop=True)
display(df.head(), df.tail())

## 2. Controles automáticos de calidad de la serie agregada

In [ ]:
esperados = pd.date_range('2012-01-01','2025-12-01',freq='MS')
assert len(df)==168, f'Se esperaban 168 registros y hay {len(df)}'
assert df.ds.is_unique, 'Hay meses duplicados'
assert df.ds.tolist()==esperados.tolist(), 'Faltan meses o la frecuencia no es mensual'
assert df.y.notna().all() and np.isfinite(df.y).all(), 'Existen valores faltantes o no finitos'
assert (df.y>=0).all(), 'Se encontraron producciones negativas'
control_anual = pd.read_excel(ARCHIVO,sheet_name='Serie_anual')
anuales = df.assign(Año=df.ds.dt.year).groupby('Año')['y'].sum()
control = pd.DataFrame({'Calculado_m3':anuales, 'Excel_m3':control_anual.set_index('Año')['Producción anual (m³)']})
control['Diferencia_m3']=control.Calculado_m3-control.Excel_m3
assert np.allclose(control.Calculado_m3,control.Excel_m3,atol=0.1), 'Diferencia con serie anual'
print('Control temporal y anual: OK');display(control)
control.to_csv(SALIDA/'control_totales_anuales.csv',encoding='utf-8-sig')

## 3. Análisis exploratorio 2012–2025
Etapas descriptivas: 2012–2017, 2018–2022 y 2023–2025. No representan rupturas estructurales estimadas formalmente.

In [ ]:
fig, ax=plt.subplots(figsize=(12,4))
ax.plot(df.ds,df.y/1e6,lw=1.7,label='Producción mensual')
ax.plot(df.ds,df.y.rolling(12,min_periods=12).mean()/1e6,lw=2,label='Media móvil 12 meses')
for fecha in ['2018-01-01','2023-01-01']:
    ax.axvline(pd.Timestamp(fecha),color='grey',ls='--',alpha=.6)
ax.set(title='Vaca Muerta — petróleo mensual, 2012–2025',ylabel='Millones de m³',xlabel='Fecha')
ax.grid(alpha=.2);ax.legend();fig.tight_layout()
fig.savefig(SALIDA/'01_evolucion_mensual.png',dpi=180);plt.show()
fig,ax=plt.subplots(figsize=(11,4))
ax.bar(anuales.index,anuales.values/1e6)
ax.set(title='Producción petrolera anual',xlabel='Año',ylabel='Millones de m³')
ax.grid(axis='y',alpha=.2);fig.tight_layout()
fig.savefig(SALIDA/'02_totales_anuales.png',dpi=180);plt.show()

## 4. Métricas y procedimientos reproducibles
Sesgo = promedio de (pronóstico − observado), negativo indica subestimación. MAPE requiere valores reales diferentes de cero.

In [ ]:
def metricas(real,pred):
    r=np.asarray(real,dtype=float); p=np.asarray(pred,dtype=float)
    assert len(r)==len(p) and np.isfinite(r).all() and np.isfinite(p).all()
    return {'MAPE_%':float(np.mean(np.abs((r-p)/r))*100) if (r!=0).all() else np.nan,
            'MAE_m3':float(mean_absolute_error(r,p)),
            'RMSE_m3':float(np.sqrt(mean_squared_error(r,p))),
            'Sesgo_m3':float(np.mean(p-r))}

def modelo_prophet(cps=0.03,sps=0.1,estacionalidad=True):
    return Prophet(growth='linear', yearly_seasonality=estacionalidad,
                   weekly_seasonality=False,daily_seasonality=False,
                   changepoint_prior_scale=cps,seasonality_prior_scale=sps,
                   interval_width=0.95)

def evaluar(inicio,anio_test,cps=0.03,sps=0.1,estacionalidad=True,devolver_series=False):
    train=df[(df.ds>=pd.Timestamp(f'{inicio}-01-01')) & (df.ds<pd.Timestamp(f'{anio_test}-01-01'))].copy()
    test=df[(df.ds>=pd.Timestamp(f'{anio_test}-01-01')) & (df.ds<pd.Timestamp(f'{anio_test+1}-01-01'))].copy()
    assert len(train)>=24 and len(test)==12, 'Revisar las fechas de train/test'
    m=modelo_prophet(cps,sps,estacionalidad);m.fit(train)
    fc=m.predict(test[['ds']])
    serie=test[['ds','y']].merge(fc[['ds','yhat','yhat_lower','yhat_upper']],on='ds',validate='one_to_one')
    fila={'Inicio':inicio,'Test':anio_test,'Estacionalidad':estacionalidad,
          'CPS':cps,'SPS':sps,'N_train':len(train), **metricas(serie.y,serie.yhat)}
    return (fila,serie,m) if devolver_series else fila

## 5. Comparación de ventanas de entrenamiento
Evaluar 2024 exclusivamente con datos hasta 2023; evaluar 2025 con datos hasta 2024. Este análisis es exploratorio; **no elegir retrospectivamente el mejor modelo mirando 2025 y denominarlo test independiente**.

In [ ]:
resultados_ventanas=[]
for anio_test in [2024,2025]:
    for inicio in [2018,2020,2023]:
        if anio_test-inicio>=2:
            for seasonal in [False,True]:
                resultados_ventanas.append(evaluar(inicio,anio_test,estacionalidad=seasonal))
ventanas=pd.DataFrame(resultados_ventanas)
display(ventanas.sort_values(['Test','MAPE_%']))
ventanas.to_csv(SALIDA/'03_comparacion_ventanas.csv',index=False,encoding='utf-8-sig')

## 6. Búsqueda de parámetros con validación 2024
La malla siguiente utiliza solo la evaluación de 2024 para explorar CPS/SPS. Para respetar la comparación acordada en el paper, el modelo final de referencia queda **fijado explícitamente** en CPS=0,03 y SPS=0,1, aunque una búsqueda más amplia arrojara otros parámetros.

In [ ]:
grid=[]
for cps in [0.01,0.03,0.05]:
    for sps in [0.1,1.0,5.0,10.0]:
        grid.append(evaluar(2020,2024,cps=cps,sps=sps,estacionalidad=True))
resultados_grid=pd.DataFrame(grid).sort_values(['MAPE_%','RMSE_m3']).reset_index(drop=True)
display(resultados_grid)
resultados_grid.to_csv(SALIDA/'04_grid_validacion_2024.csv',index=False,encoding='utf-8-sig')
CPS_FINAL=0.03
SPS_FINAL=0.1
ESTACIONALIDAD_FINAL=True
print('Configuración fijada:',CPS_FINAL,SPS_FINAL,ESTACIONALIDAD_FINAL)

## 7. Backtesting de referencia 2024 y test 2025
Los parámetros se mantienen constantes en ambos años. Los valores impresos sirven para contrastar los resultados del paper: MAPE 2024 ≈ 3,921 % y MAPE 2025 ≈ 7,182 %. Diferencias de versión de Prophet, datos o especificación deben reportarse, no ocultarse.

In [ ]:
val24,serie24,m24=evaluar(2020,2024,CPS_FINAL,SPS_FINAL,ESTACIONALIDAD_FINAL,True)
val25,serie25,m25=evaluar(2020,2025,CPS_FINAL,SPS_FINAL,ESTACIONALIDAD_FINAL,True)
resumen_backtesting=pd.DataFrame([val24,val25]);display(resumen_backtesting)
resumen_backtesting.to_csv(SALIDA/'05_metricas_2024_2025.csv',index=False,encoding='utf-8-sig')
serie24.to_csv(SALIDA/'06_predicciones_test_2024.csv',index=False,encoding='utf-8-sig')
serie25.to_csv(SALIDA/'07_predicciones_test_2025.csv',index=False,encoding='utf-8-sig')
for anio,serie in [(2024,serie24),(2025,serie25)]:
    fig,ax=plt.subplots(figsize=(10,3.6))
    ax.plot(serie.ds,serie.y/1e6,'o-',label='Real');ax.plot(serie.ds,serie.yhat/1e6,'o--',label='Prophet')
    ax.fill_between(serie.ds,serie.yhat_lower/1e6,serie.yhat_upper/1e6,alpha=.12,label='Intervalo predictivo puntual')
    ax.set(title=f'Backtesting {anio}: real vs pronosticado',ylabel='Millones de m³')
    ax.grid(alpha=.2);ax.legend();fig.tight_layout()
    fig.savefig(SALIDA/f'08_backtesting_{anio}.png',dpi=180);plt.show()

## 8. Diagnóstico de errores y residuos
Ljung–Box prueba autocorrelación serial. El diagnóstico se evalúa separadamente en cada período. La presencia de autocorrelación limita la suficiencia del modelo.

In [ ]:
diagnosticos=[]
for anio,serie in [(2024,serie24),(2025,serie25)]:
    resid=serie.y-serie.yhat
    lb=acorr_ljungbox(resid,lags=[1,3,6],return_df=True)
    lb.insert(0,'Año',anio);diagnosticos.append(lb.reset_index(names='Lag'))
    fig,ax=plt.subplots(figsize=(10,3))
    ax.axhline(0,color='black',lw=1);ax.plot(serie.ds,resid/1000,'o-')
    ax.set(title=f'Residuos del pronóstico {anio} (real − pronóstico)',ylabel='Miles de m³')
    ax.grid(alpha=.2);fig.tight_layout();fig.savefig(SALIDA/f'09_residuos_{anio}.png',dpi=180);plt.show()
    sem=serie.assign(Semestre=np.where(serie.ds.dt.month<=6,'S1','S2')).groupby('Semestre').agg(Real=('y','sum'),Predicho=('yhat','sum'))
    sem['Diferencia_relativa_%']=(sem.Predicho/sem.Real-1)*100
    print('Semestres',anio);display(sem)
diagnostico_lb=pd.concat(diagnosticos,ignore_index=True)
display(diagnostico_lb)
diagnostico_lb.to_csv(SALIDA/'10_ljung_box_residuos.csv',index=False,encoding='utf-8-sig')

## 9. Modelo reentrenado 2020–2025 y pronóstico 2026
Una vez documentado el test, se reentrena el modelo de parámetros fijos con los 72 meses de 2020–2025. Esta es una operación distinta del test 2025, entrenado con 60 meses. **Atención:** el total resultante puede diferir del informado en la ejecución final del 06/10.

In [ ]:
train_final=df[(df.ds>=pd.Timestamp('2020-01-01'))&(df.ds<=pd.Timestamp('2025-12-01'))].copy()
assert len(train_final)==72
modelo_final=modelo_prophet(CPS_FINAL,SPS_FINAL,ESTACIONALIDAD_FINAL)
modelo_final.fit(train_final)
futuro=pd.DataFrame({'ds':pd.date_range('2026-01-01','2026-12-01',freq='MS')})
fc=modelo_final.predict(futuro)
forecast_2026=fc[['ds','yhat','yhat_lower','yhat_upper']].copy()
forecast_2026.columns=['Fecha','Pronostico_m3','Limite_inferior_puntual_95','Limite_superior_puntual_95']
total_2026=float(forecast_2026.Pronostico_m3.sum())
print(f'Pronóstico anual REPRODUCIDO: {total_2026:,.2f} m³')
display(forecast_2026)
forecast_2026.to_excel(SALIDA/'11_forecast_mensual_2026.xlsx',index=False)
fig,ax=plt.subplots(figsize=(12,4))
ax.plot(df[df.ds>='2023-01-01'].ds,df[df.ds>='2023-01-01'].y/1e6,label='Histórico 2023–2025')
ax.plot(forecast_2026.Fecha,forecast_2026.Pronostico_m3/1e6,'o-',label='Pronóstico 2026')
ax.fill_between(forecast_2026.Fecha,forecast_2026.Limite_inferior_puntual_95/1e6,forecast_2026.Limite_superior_puntual_95/1e6,alpha=.12,label='Bandas puntuales 95 %')
ax.set(title='Evolución reciente y pronóstico 2026',ylabel='Millones de m³')
ax.grid(alpha=.2);ax.legend();fig.tight_layout();fig.savefig(SALIDA/'12_forecast_2026.png',dpi=180);plt.show()

## 10. Conciliación con el paper y límites de interpretación
La cifra 35.596.087,84 m³ se conserva como **referencia histórica reportada**, no como cálculo confirmado en este Notebook. Si no coincide, revisar versión del paquete Prophet, semillas/aleatoriedad para bandas, insumos y especialmente si la ejecución histórica proyectó 2026 entrenando hasta 2024 o reentrenó incluyendo 2025. No alterar parámetros a posteriori simplemente para forzar la cifra. Las bandas inferiores y superiores mensuales no constituyen por suma un intervalo anual calibrado.

In [ ]:
REFERENCIA_PAPER_M3=35_596_087.84
diferencia=total_2026-REFERENCIA_PAPER_M3
conciliacion=pd.DataFrame([{'Concepto':'Total pronosticado 2026 (modelo reentrenado 2020–2025)', 'm3':total_2026},
                            {'Concepto':'Referencia comunicada 06/10/2026','m3':REFERENCIA_PAPER_M3},
                            {'Concepto':'Diferencia (reproducido menos comunicado)','m3':diferencia}])
display(conciliacion)
conciliacion.to_csv(SALIDA/'13_conciliacion_total_2026.csv',index=False,encoding='utf-8-sig')
if abs(diferencia)>1:
    print('ATENCIÓN: el forecast no coincide con la cifra comunicada. Revisar especificaciones antes de declarar reproducibilidad total.')
else:
    print('Conciliación del total: OK (tolerancia 1 m³).')

## 11. Inventario y alcance de los entregables
- `VM_Serie_Fuente_2012_2025.xlsx`: serie consolidada de 168 meses; contiene controles de calidad y criterios de tratamiento.
- `VM_Notebook_Maestro_Completo_2026.ipynb`: este Notebook maestro, con flujo íntegro **desde la serie agregada**.
- `VM_Notebook_Historico_01oct2026.ipynb`: respaldo sin editar de la versión previa.
- Carpeta `salidas_VM_2026`: se crea al ejecutar y contiene métricas, tablas y gráficos.

**Límite de trazabilidad:** para reproducir la depuración desde los registros de pozo se requieren además los CSV anuales nacionales 2012–2025, sus filtros y scripts ETL originales; dichos archivos no forman parte de esta entrega. No confundir “Excel fuente del modelado” con “registros brutos oficiales”.